In [1]:
import pandas as pd
import os,glob,math
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt


In [2]:
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
plt.rc("font",family="Arial")

In [3]:
from statannotations.Annotator import Annotator
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [4]:
# pd.set_option('display.max_columns', 10)
pd.set_option('display.max_rows', 10)

In [5]:
dfall = pd.read_csv("all_neurons_sholl_geno.csv",index_col=0)

In [8]:
df = dfall[["50","100","cluster"]]

In [9]:
df.columns = ["50.0","100.0","group"]

In [10]:
df

,50.0,100.0,group
251034_010_Vglut1,26,30,Vglut1
234172_015_Crh,19,8,Crh
221058_038_Sst,19,14,Sst
221058_107_Sst,9,14,Sst
221297_038_Sst,19,16,Sst
...,...,...,...
221058_113_Sst,7,19,Sst
251034_003_Vglut1,17,31,Vglut1
251034_004_Vglut1,30,36,Vglut1
234172_034_Crh,4,8,Crh


In [12]:
# for 4 groups, we can use the following code ['Sst',"Crh","Vglut1"]

for para in ["50.0","100.0"]:


    # 计算 Tukey HSD 的 p 值

    tukey = pairwise_tukeyhsd(df[para], df['group'])
    summary = pd.DataFrame(data=tukey.summary().data[1:], columns=tukey.summary().data[0])
    summary["p-adj"] = summary["p-adj"].astype(float)
    summary.to_csv("sholl_results_of_%s.csv"%(para))
    group_list = []
    for j in summary.iterrows():
        group_list.append((j[1]["group1"],j[1]["group2"]))

    p_values = []
    for g1, g2 in group_list:
        result = summary.query(f"group1 == '{g1}' and group2 == '{g2}'")
        if not result.empty:
            p_values.append(result["p-adj"].values[0])
        else:
            p_values.append(summary.query(f"group1 == '{g2}' and group2 == '{g1}'")["p-adj"].values[0])  # 或者用默认值


    # 开始绘图
    plt.figure(figsize=(4, 6))

    sns.stripplot(data=df, x="group", y=para,order=['Sst',"Crh","Vglut1"],
                jitter=0.2,
                edgecolor="black", 
                palette=["white", "white", "white"], 
                size=9, linewidth=1.5,
    )

    sns.barplot(data=df, x="group", y=para, order=['Sst',"Crh","Vglut1"],            
                palette=["#0000ff", "red", "#107010"],
                errorbar="se",
                capsize=0.4, errcolor="0", errwidth=2,
                edgecolor="0", 
                linewidth=1.5)

    # 设置注释的配对组
    pairs = group_list

    # 添加统计学显著性注释
    annotator = Annotator(plt.gca(), pairs, data=df, 
                        x="group", y=para, 
                        order=['Sst',"Crh","Vglut1"],
                        )
    annotator.configure(test=None, text_format="simple", loc="inside", verbose=0, fontsize=16,pvalue_format_string="{:.3f}",
                        line_width=2)
    annotator.set_pvalues(p_values).annotate()

    # 定制图表
    # plt.legend(title="Group", bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0)
    plt.xlabel("")
    plt.ylabel(para, fontsize=16)
    plt.tick_params(axis='x', labelsize=16)
    plt.xticks(rotation=30,ha='right', va='top')  
    plt.tick_params(axis='y', labelsize=16)
    plt.tight_layout()
    plt.savefig("Sholl_%s_geno.pdf"%(para),dpi=300)
    plt.savefig("Sholl_%s_geno.jpg"%(para),dpi=300)
    plt.close()



C:\Users\zljia\AppData\Local\Temp\ipykernel_2168\1076261683.py:28: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(data=df, x="group", y=para,order=['Sst',"Crh","Vglut1"],
C:\Users\zljia\AppData\Local\Temp\ipykernel_2168\1076261683.py:35: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x="group", y=para, order=['Sst',"Crh","Vglut1"],
C:\Users\zljia\AppData\Local\Temp\ipykernel_2168\1076261683.py:35: FutureWarning: 

The `errcolor` parameter is deprecated. And will be removed in v0.15.0. Pass `err_kws={'color': '0'}` instead.

  sns.barplot(data=df, x="group", y=para, order=['Sst',"Crh","Vglut1"],
C:\Users\zljia\AppData\Local\Temp\ipykernel_2168\1076261683.py:35: FutureWarning: 

The